In [1]:
from segmentiq import *

In [2]:



my_search_grid = {
    "min_sample_size": [10000,5000, 1000],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="is_defaulted",
    # n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.1,
    min_events = 5,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None,
    ignore_features=['application_id'],
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/loan_applications_500k.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-12 13:36:02,026 | INFO     | [builder.py:337] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-12 13:36:02,536 | INFO     | [builder.py:352] | Dynamic Grid Search Enabled: 6 total configurations.
2026-07-12 13:36:02,543 | INFO     | [builder.py:373] | Iteration 1 | Remaining Volume: 500,000 | Base Rate: 15.01%
2026-07-12 13:36:08,324 | INFO     | [builder.py:509] | Feature Usage Tracker Update -> 'credit_score' used count = 1
2026-07-12 13:36:08,325 | INFO     | [builder.py:509] | Feature Usage Tracker Update -> 'is_auto_pay' used count = 1
2026-07-12 13:36:08,325 | INFO     | [builder.py:509] | Feature Usage Tracker Update -> 'debt_to_income_ratio' used count = 1
2026-07-12 13:36:08,325 | INFO     | [builder.py:524] | Segment 1 Captured (Size Floor: 1000 | Lift Floor: 2.0): credit_score < 582.31 AND is_auto_pay < 0.50 AND debt_to_income_ratio >= 60.80
2026-07-12 13:36:09,034 | INFO     | [builder.py:373] | Iteration 2 | Remaining Volume: 498,210 | Base Rate: 14.73%
20

In [3]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate   |        lift        |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
|   0.0   |   408133.0  |    48334.0    | 11.842708136808344 | 15.009400000000001 | 81.62660000000001 | 0.7890194236150908 |
|   1.0   |    1790.0   |     1675.0    | 93.57541899441341  | 15.009400000000001 |       0.358       | 6.234454341573508  |
|   2.0   |   90077.0   |    25038.0    | 27.796218790590274 | 15.009400000000001 |      18.0154      | 1.8519207157241644 |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+


In [4]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   credit_score=(-inf, 582.31) & is_auto_pay=(-inf, 0.50) & debt_to_income_ratio=[60.80, inf)
SQL Filter: credit_score < 582.31 AND is_auto_pay < 0.50 AND debt_to_income_ratio >= 60.80
--------------------------------------------------
Segment ID: 2
Raw Rule:   annual_income=(-inf, 10000.13)
SQL Filter: annual_income < 10000.13
--------------------------------------------------


In [5]:
builder.explain_feature_journey("recovery_collection_fee")

AUDIT TRAIL FOR FEATURE: 'recovery_collection_fee'

[Iteration 1]
  • Current dynamic IV   : 0.0000
  • Previous times used  : 0
  • Selection Status     : Excluded (Information Value is Zero/Invalid)
  • Winner this round    : credit_score=(-inf, 582.31) & is_auto_pay=(-inf, 0.50) & debt_to_income_ratio=[60.80, inf) (Variables: ['credit_score', 'is_auto_pay', 'debt_to_income_ratio'])

[Iteration 2]
  • Current dynamic IV   : 0.0000
  • Previous times used  : 0
  • Selection Status     : Excluded (Information Value is Zero/Invalid)
  • Winner this round    : annual_income=(-inf, 10000.13) (Variables: ['annual_income'])

[Iteration 3]
  • Current dynamic IV   : 0.0000
  • Previous times used  : 0
  • Selection Status     : Excluded (Information Value is Zero/Invalid)
